In [ ]:
from datetime import date

import pandas as pd
import numpy as np
import holidays
import yaml
from sqlalchemy import create_engine, text


# database connections 


In [ ]:
with open('../../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['OLTP']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)


Para inicializar la dimension creamos un dataframe donde vamos a añadir las fechas y demas campos


In [ ]:
dim_fecha = pd.DataFrame({
    "date": pd.date_range(start='1/1/2023', end='31/12/2025', freq='D')
})
dim_fecha.head()


vamos a añadir algunas columnas como lo son el año, mes, el dia, el dia de la semana y en que quarto del año


In [ ]:
dim_fecha["year"] = dim_fecha["date"].dt.year
dim_fecha["month"] = dim_fecha["date"].dt.month
dim_fecha["day"] = dim_fecha["date"].dt.day
dim_fecha["weekday"] = dim_fecha["date"].dt.weekday
dim_fecha["quarter"] = dim_fecha["date"].dt.quarter
dim_fecha["day_of_year"] = dim_fecha["date"].dt.day_of_year
dim_fecha["day_of_month"] = dim_fecha["date"].dt.days_in_month
dim_fecha["month_str"] = dim_fecha["date"].dt.month_name()
dim_fecha["day_str"] = dim_fecha["date"].dt.day_name()
dim_fecha["date_str"] = dim_fecha["date"].dt.strftime("%d/%m/%Y")
co_holidays = holidays.CO(language="es")
dim_fecha["is_Holiday"] = dim_fecha["date"].apply(lambda x: x in co_holidays)
dim_fecha["holiday"] = dim_fecha["date"].apply(lambda x: co_holidays.get(x))
dim_fecha["holiday"] = dim_fecha["holiday"].fillna("NO APLICA")
dim_fecha["weekend"] = dim_fecha["weekday"].apply(lambda x: x > 4)
dim_fecha["saved"] = date.today()
dim_fecha.head()


# load


In [ ]:
with etl_conn.begin() as conn:
    conn.execute(text('Delete from dim_fecha'))
dim_fecha.to_sql('dim_fecha', etl_conn, if_exists='append', index=False)
